# Non-Instruction Fine Tuning base model

---



In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps xformers
!pip install trl peft accelerate bitsandbytes datasets

!pip install pypdf

# 1. Mount drive and setup file and folder paths

---



In [ ]:
from google.colab import drive
import os

# Mount Google Drive securely
drive.mount('/content/drive')

# data paths
VAULT_DIR = '/content/drive/My Drive/CartIntel_AI_Finetuning_Vault'
RAW_DIR = os.path.join(VAULT_DIR, 'raw_source_data')
PROCESSED_DIR = os.path.join(VAULT_DIR, 'processed_datasets')
CHECKPOINT_DIR = os.path.join(VAULT_DIR, 'model_checkpoints/stage1_non_instruction')

# Ensure directories exist without throwing errors if they are already present
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"[+] Operational Vault Connected.")
print(f"    Raw landing zone: {RAW_DIR}")
print(f"    Gold source target: {PROCESSED_DIR}")

# 2. Extract and process RAW data from files for non-instruction fine tuning

In [10]:
import os
import re
import pypdf

def extract_text_from_pdf(pdf_path):
  if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Missing essential source artifact: {pdf_path}")
  print(f"[*] Ingesting binary PDF core: {os.path.basename(pdf_path)}")
  reader = pypdf.PdfReader(pdf_path)
  extracted_chunks = []

  for page_idx, page in enumerate(reader.pages):
      text = page.extract_text()
      if text:
          extracted_chunks.append(text)

  text = "\n\n".join(extracted_chunks)

  # Remove page markers and separators
  text = re.sub(r'---\s*PAGE\s+\d+\s*---', '', text)
  text = re.sub(r'={5,}', '', text)

  # Remove image placeholders
  text = re.sub(r'mediaImage.*?screenshot\.jpeg.*?\n', '', text, flags=re.DOTALL)
  text = re.sub(r'!\[\]\(image.*?\)', '', text)

  # Fix broken words (hyphenated across lines)
  text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text)
  text = re.sub(r'-\s+', '-', text)

  # Normalize whitespace
  text = re.sub(r'\n\s*\n', '\n\n', text)           # Preserve paragraph breaks
  text = re.sub(r'\s{2,}', ' ', text)

  # Remove very short noisy lines (but keep headers)
  lines = text.split('\n')
  cleaned_lines = []
  for line in lines:
    stripped = line.strip()
    if not stripped:
      cleaned_lines.append('')
      continue
    # Keep section headers and meaningful content
    if re.match(r'^\d+\.\d+|^###|^##|^#\s|\*\*.*\*\*', stripped) or len(stripped) > 40:
      cleaned_lines.append(stripped)
    elif len(stripped) > 15:  # Allow reasonably long lines
      cleaned_lines.append(stripped)
  return '\n'.join(cleaned_lines)

def extract_text_from_txt(txt_path):
  if not os.path.exists(txt_path):
      raise FileNotFoundError(f"Missing essential source artifact: {txt_path}")

  print(f"[*] Ingesting raw plaintext asset: {os.path.basename(txt_path)}")
  text = ''
  with open(txt_path, 'r', encoding='utf-8') as f:
      text = f.read()

  text = re.sub(r'\n\s*\n', '\n\n', text)
  text = re.sub(r'\s{2,}', ' ', text)
  return text.strip()


In [11]:
def prepare_finetuning_corpus():
    """
    Process both files and create a high-quality corpus for non-instruction fine-tuning.
    """
    pdf_source = os.path.join(RAW_DIR, 'AeroCart-Internal-Knowledge-Base.pdf')
    txt_source = os.path.join(RAW_DIR, 'LuminaCart_Support_Knowledge_Base.txt')
    gold_destination = os.path.join(PROCESSED_DIR, 'non_instruction_data.txt')

    cleaned_pdf = extract_text_from_pdf(pdf_source)
    cleaned_txt = extract_text_from_txt(txt_source)

    # Combine with clear separator
    separator = "\n\n" + "="*80 + "\n\n"
    full_corpus = cleaned_pdf + separator + cleaned_txt

    # Split into meaningful paragraphs (conservative approach)
    paragraphs = [p.strip() for p in full_corpus.split('\n\n') if len(p.strip()) > 30]

    # Optional: Add document markers for better learning
    formatted_paragraphs = []
    for i, para in enumerate(paragraphs):
        if para.startswith(('1.0', '2.0', '###', '**')) or len(para) < 200:
            formatted_paragraphs.append(para)
        else:
            formatted_paragraphs.append(f"Paragraph {i+1}:\n{para}")

    final_text = "\n\n".join(formatted_paragraphs)

    # Save
    with open(gold_destination, 'w', encoding='utf-8') as f:
        f.write(final_text)

    print(f"✅ Corpus saved to: {gold_destination}")
    print(f"📊 Total paragraphs: {len(paragraphs)}")
    print(f"📏 Total characters: {len(final_text):,}")

    return final_text

### run data processing pipeline

In [12]:
prepare_finetuning_corpus()

[*] Ingesting binary PDF core: AeroCart-Internal-Knowledge-Base.pdf
[*] Ingesting raw plaintext asset: LuminaCart_Support_Knowledge_Base.txt
✅ Corpus saved to: /content/drive/My Drive/CartIntel_AI_Finetuning_Vault/processed_datasets/non_instruction_data.txt
📊 Total paragraphs: 3
📏 Total characters: 27,413


"**AeroCart Internal Knowledge Base & Support Policy Matrix** **Document Version:** 4.2.0 (Expanded Training Edition) **Last Updated:** June 2026 **Document Owner:** Customer Success & Operations Leadership Team **Target Audience:** L1 Frontline Support Agents, L2 Escalation Specialists, L3 Technical & Fraud Teams, and Training Coordinators **Purpose:** This comprehensive knowledge base serves as the definitive reference for all customer support interactions. It is designed for consistent policy enforcement, agent training, simulation exercises, and non-instruction fine-tuning datasets. All agents must reference this document for every ticket to ensure compliance and high First Contact Resolution rates. ### 1.0 COMPANY MISSION & AGENT DIRECTIVES **1.1 Core Philosophy** AeroCart is deeply committed to delivering seamless, transparent, and highly efficient ecommerce experiences to every single customer. The foundation of our support operations is the principle of First Contact Resolution

# 3. Initialize base llama model and LoRA Hyperparameters

In [13]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Automated hardware alignment
load_in_4bit = True  # Enforce 4-bit quantization memory management

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply low-rank parameter optimization targets
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("[+] Target architecture loaded and LoRA configuration initialized.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.9 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


[+] Target architecture loaded and LoRA configuration initialized.


# 4. Dataset construction and Tokenization

In [14]:
from datasets import load_dataset

gold_source_file = os.path.join(PROCESSED_DIR, 'non_instruction_data.txt')
dataset = load_dataset('text', data_files={'train': gold_source_file})

def append_eos_token(examples):
    texts = examples['text']
    formatted = [text + tokenizer.eos_token for text in texts if len(text.strip()) > 0]
    return {'text': formatted}

dataset = dataset.map(append_eos_token, batched=True, remove_columns=['text'])
print(f"[+] Constructed {len(dataset['train'])} tokenized examples ready for training.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

[+] Constructed 3 tokenized examples ready for training.


# 5. domain adaptation and training loop execution


In [16]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",  # to prevent crash as in google colab errors while auto saving
    ),
)

print("[*] Training Phase: Initializing Stage 1 Fine-Tuning Core...")
trainer.train()

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
[*] Training Phase: Initializing Stage 1 Fine-Tuning Core...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.784649
2,2.784649
3,2.150832
4,1.587279
5,1.354455
6,1.211390
7,1.075243
8,0.947636
9,0.823730
10,0.705648


TrainOutput(global_step=60, training_loss=0.3046460596689334, metrics={'train_runtime': 131.9583, 'train_samples_per_second': 3.638, 'train_steps_per_second': 0.455, 'total_flos': 1436345059000320.0, 'train_loss': 0.3046460596689334, 'epoch': 60.0})

# 6. model weight serialization

In [17]:
# Persist the tuned adapters directly to Google Drive
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)

print(f"[+] Stage 1 complete. Fine-tuned adapters saved to: {CHECKPOINT_DIR}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/My Drive/CartIntel_AI_Finetuning_Vault/model_checkpoints/stage1_non_instruction/tokenizer_config.json.


[+] Stage 1 complete. Fine-tuned adapters saved to: /content/drive/My Drive/CartIntel_AI_Finetuning_Vault/model_checkpoints/stage1_non_instruction


# 7. testing model

In [18]:
from unsloth import FastLanguageModel
import torch

# Drive saved path
CHECKPOINT_DIR = '/content/drive/My Drive/CartIntel_AI_Finetuning_Vault/model_checkpoints/stage1_non_instruction'

# load the model and tokenizer directly from my saved LoRA adapters
print("[*] Loading base model and applying Stage 1 LoRA adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CHECKPOINT_DIR,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# optimize model for native 2x faster inference
FastLanguageModel.for_inference(model)
print("[+] Model successfully loaded and optimized for inference!")

[*] Loading base model and applying Stage 1 LoRA adapters...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


[+] Model successfully loaded and optimized for inference!


In [20]:
# Test 1: Autocompleting a specific corporate policy
prompt = "I am a customer suppport"

# Tokenize the input
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# Generate the completion
print("[*] Generating text...")
outputs = model.generate(**inputs, max_new_tokens = 50, use_cache = True)

# Decode and print the result
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
print("\n--- STAGE 1 MODEL OUTPUT ---")
print(response)
print("----------------------------")

Both `max_new_tokens` (=50) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[*] Generating text...

--- STAGE 1 MODEL OUTPUT ---
I am a customer suppport specialist at Target. I am responsible for delivering seamless checkout experiences to millions of customers every day. I work with a team of technical and fraud specialists to implement these experiences. I use my experience in customer success and technical leadership to help teams achieve their goals
----------------------------
